In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch11-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "11-06-limits-of-explanations.py"

# 在硝基苯上加羥基，哪個位置值得複測？

DeepChem 首次執行會下載 ESOL。四個指定結構另行保留，讓我們比較簡單模型的係數與同骨架的實測差值。 做小練習時，把程式加在下一格末尾，再重跑該格和最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import deepchem as dc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from sklearn.linear_model import LinearRegression

_, (data,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False,
)
phenol = Chem.MolFromSmarts("[c]-[O;H1;+0]")
def canonical(smiles):
    return Chem.MolToSmiles(Chem.MolFromSmiles(smiles))
table = pd.DataFrame({
    "structure": [canonical(str(s)) for s in data.ids],
    "logS": data.y[:, 0],
    "OH": [int(Chem.MolFromSmiles(str(s)).HasSubstructMatch(phenol)) for s in data.ids],
})
series = {
    "Parent": "O=[N+]([O-])c1ccccc1",
    "ortho": "O=[N+]([O-])c1ccccc1O",
    "meta": "O=[N+]([O-])c1cccc(O)c1",
    "para": "O=[N+]([O-])c1ccc(O)cc1",
}
keys = {name: canonical(s) for name, s in series.items()}
train = table[~table.structure.isin(keys.values())]
held = table[table.structure.isin(keys.values())].set_index("structure")
assert len(held) == 4 and held.index.is_unique
model = LinearRegression().fit(train[["OH"]], train["logS"])
coefficient = model.coef_[0]
print(train.groupby("OH")["logS"].agg(["count", "mean"]).round(4))
print(f"Model difference = {coefficient:+.4f} log units")
parent = held.loc[keys["Parent"], "logS"]
print(f"Parent logS = {parent:.2f}")
labels = ["ortho", "para"]
difference = []
for label in labels:
    observed = held.loc[keys[label], "logS"]
    delta = observed - parent
    difference.append(delta)
    print(label, f"logS = {observed:.2f}, difference = {delta:+.2f}")

fig, ax = plt.subplots(figsize=(7.5, 3.4), layout="constrained")
y = np.arange(len(labels))
ax.scatter(difference, y, s=105, color="#16766f", label="ESOL pair", zorder=3)
ax.axvline(coefficient, color="#a6532e", ls="--", label="Model coefficient")
ax.axvline(0, color="#555", lw=0.8)
for i, val in enumerate(difference):
    ax.text(val+0.035, i, f"{val:+.2f}", va="center")
ax.text(coefficient+0.025, 0.47, f"{coefficient:+.2f}", color="#843f22")
ax.set(yticks=y, yticklabels=["ortho-OH", "para-OH"], xlim=(-0.1, 1.32),
       ylim=(-0.55, 1.55), xlabel="Change in log solubility relative to nitrobenzene")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.legend(loc="upper left", ncols=2, frameon=False)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.savefig(out / "11-06-limits-of-explanations.png", dpi=170)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "11-06-limits-of-explanations.png")))